In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

# 載入隱藏的 .env 檔案
load_dotenv()
TDX_CLIENT_ID = os.getenv("TDX_CLIENT_ID")
TDX_CLIENT_SECRET = os.getenv("TDX_CLIENT_SECRET")

In [3]:
auth_url = "https://tdx.transportdata.tw/auth/realms/TDXConnect/protocol/openid-connect/token"
auth_data = {
    'grant_type': 'client_credentials',
    'client_id': TDX_CLIENT_ID,
    'client_secret': TDX_CLIENT_SECRET
}

response = requests.post(auth_url, data=auth_data)

if response.status_code == 200:
    access_token = response.json().get('access_token')
    print("✅ 成功取得 Access Token！")
else:
    print(f"❌ 取得 Token 失敗: {response.text}")

✅ 成功取得 Access Token！


In [ ]:
# [Cell 3] 抓取台南市公車預估到站資料
api_url = "https://tdx.transportdata.tw/api/basic/v2/Bus/EstimatedTimeOfArrival/City/Tainan?%24format=JSON"

# 把通行證塞進請求標頭
headers = {'authorization': f'Bearer {access_token}'}

res = requests.get(api_url, headers=headers)

if res.status_code == 200:
    data = res.json()
    print(f"✅ 成功抓取！當前台南市共有 {len(data)} 筆動態資料。")
else:
    print(f"❌ API 呼叫失敗，狀態碼: {res.status_code}")

正在呼叫 TDX API...
✅ 成功抓取！當前台南市共有 756 筆動態資料。


In [7]:
# [Cell 4] 整理成 DataFrame 並篩選特定路線
bus_data = []
for item in data:
    bus_data.append({
        'Route_Name': item.get('RouteName', {}).get('Zh_tw'), # 路線名稱
        'Stop_Name': item.get('StopName', {}).get('Zh_tw'),   # 站牌名稱
        'Direction': '去程' if item.get('Direction') == 0 else '返程',
        'Estimate_Time_sec': item.get('EstimateTime'),        # 預估到站時間(秒)
        'Plate_Numb': item.get('PlateNumb')                   # 車牌
    })

# 轉換成表格
df = pd.DataFrame(bus_data)
df.head()  # 顯示前幾筆資料確認格式
# # --- 這裡開始可以自由修改 ---
# # 假設我們想觀察經過中西區的「88」號觀光公車 (你也可以改成 '2', '99' 等等)
# target_route = "88"
# df_target = df[df['Route_Name'] == target_route].copy()

# # 剔除還沒發車(數值為 NaN)的站點
# df_target = df_target.dropna(subset=['Estimate_Time_sec'])

# # 把秒數換算成分鐘，比較好懂
# df_target['Estimate_Time_min'] = (df_target['Estimate_Time_sec'] / 60).round(1)

# # 依照即將到站的時間，由近到遠排序
# df_target = df_target.sort_values(by='Estimate_Time_sec')

# print(f"\n🚍 【{target_route}】號公車 即將到站清單：")
# # 在 Jupyter 中，用 display() 印出 DataFrame 表格會非常漂亮
# display(df_target[['Stop_Name', 'Direction', 'Estimate_Time_min', 'Plate_Numb']].head(10))

,Route_Name,Stop_Name,Direction,Estimate_Time_sec,Plate_Numb
0,15,兵仔市,去程,0,EAL-2352
1,15,中華小東路口,去程,190,EAL-2352
2,62,仁德交流道,去程,1146,KKA-7382
3,62,高鐵台南站,去程,232,KKA-7380
4,62,仁德,去程,1275,KKA-7382
